# Chapter 9 — Communication and Motor Substitution

**Book:** *Brain–Computer Interfaces and Cognitive Computing: An Introduction*  
**Prerequisites:** Chapters 0–8.  
**Author:** Er. Rishabh Aryan

This chapter applies the loop to **replace** and **restore** (Chapter 1): spelling, cursors, wheelchairs and robots, prosthetic limbs, synthetic speech. The scientific temptation is to quote a bit-rate. The engineering duty is to ask whether Amina can finish a sentence on Tuesday, after setup, with errors she can repair.

**Learning outcomes**

1. Compute information transfer rate and state three reasons it is not a day of use.
2. Design a communication loop with an alphabet, a prior at S6, and an undo.
3. Distinguish decoder success, shared-control success, and listener comprehension.
4. Match motor substitution problems to sensor grain and to degrees of freedom the user must live with.
5. Place language models and shared control on the design sheet without letting them steal credit from S5.


---

## 9.1 Replacement is a life, not a trial

Amina needs to call for help, refuse a treatment, greet a visitor, and say that the room is cold. Those acts have different alphabets and different costs of error. A system that types slowly but never sends “yes” when she meant “no” may dominate a faster system that cannot be trusted in public.

> **Definition 9.1 (User-facing communication metric).**  
> Corrected meaning delivered per unit of *her* time, including setup, calibration, undos, and rest. Laboratory selections per minute are not this metric.

> **Definition 9.2 (User-facing motor metric).**  
> Tasks completed as she would count them — drink reached, icon opened, wheelchair docked — with assistance from S6 named.

If a paper reports only offline accuracy, it has not entered this chapter.


---

## 9.2 Information transfer rate, then its limits

A classical formula for a discrete $N$-class task with accuracy $P$ and decision time $T$ seconds is

$$
\mathrm{ITR} = \frac{60}{T}\left(\log_2 N + P\log_2 P + (1-P)\log_2\frac{1-P}{N-1}\right)
\quad \text{bits/min},
$$

when $0 < P \le 1$ and $P \ne 1/N$. At chance $P=1/N$ the parenthesis is zero.

The formula is useful as a back-of-envelope comparison between two spellers in the same room. It is not a life.


In [ ]:
import numpy as np


def itr_bits_per_min(N, P, T):
    P = np.clip(P, 1e-9, 1 - 1e-9)
    if abs(P - 1.0 / N) < 1e-12:
        return 0.0
    b = np.log2(N) + P * np.log2(P) + (1 - P) * np.log2((1 - P) / (N - 1))
    return (60.0 / T) * b


print("6x6 matrix, P=0.90, T=8 s  ->", round(itr_bits_per_min(36, 0.90, 8), 2), "bits/min")
print("2-class switch, P=0.90, T=2 s ->", round(itr_bits_per_min(2, 0.90, 2), 2), "bits/min")
print("36-class, P=chance, T=8 s   ->", round(itr_bits_per_min(36, 1/36, 8), 2), "bits/min")
print()
print("ITR ignores setup, undos, vocabulary, and whether the letter was the one she meant")
print("after the language model 'helped'.")


**Three reasons ITR misleads**

1. **$T$ is not the day.** Setup, gel, calibration, and rest vanish.  
2. **Errors are not symmetric.** Sending “DNR” instead of “DRINK” is not $1-P$ of a bit.  
3. **S6 is inside $P$.** A language model that forces English raises $P$ and ITR while shrinking what Amina can say.

Report ITR if you wish. Report Definition 9.1 always.


---

## 9.3 Communication loops

### Alphabets

Yes/no. Row–column matrices. Frequency-tagged targets. Keyboards with word prediction. Constrained vocabularies for attempted speech. Free speech.

Smaller alphabets are faster to decide and slower to say anything new. A yes/no loop that Amina can run when exhausted is part of a complete design, not a failure of ambition.

### Undo and repair

Every communication loop needs a cheap “that was wrong.” An error-related potential can help (Chapter 5). A residual blink or a dwell on an UNDO cell is often more reliable. The undo is Stage S6 and must be in the latency budget.

### Priors

Language models, topic lists, and “the nurse just asked a yes/no question” are cognitive computing at S6 (Definition 8.1). They are legitimate. They must be ablated when you quote a neural rate (Chapters 5 and 7).

### Channels other than vision

Auditory and tactile spellers exist for users who cannot look. They are usually slower. They are not optional afterthoughts if Amina cannot fixate.

### Comparison class

Gaze tracking, a partner-assisted letter board, and a residual-muscle switch. If any of these meets Definition 9.1 better, the BCI is research. That sentence is allowed in this book.


---

## 9.4 Motor substitution loops

### Cursors and click

Two- or three-class imagery or a continuous score mapped to velocity. Degrees of freedom are few. Shared control that snaps to icons is common. Disclose the snap.

### Wheelchairs and robots

Safety constraints at S6 are ethically required (Exercise 4.4). A decoder that can request “forward” is not a licence to ignore walls. Evaluation in an empty corridor is not evaluation in a ward.

### Prosthetic limbs and FES

High-DOF arms want implanted grain (Chapter 3) and usually want feedback. Without S8 somatosensation, users rely on vision and fatigue. Stimulation that writes touch is bidirectional and slower to justify.

### What “control” hides

A laboratory reach to a fixed target with a helper standing by is not making tea. Count the degrees of freedom the helper supplied.


---

## 9.5 Speech as substitution

Attempted-speech systems (Chapter 5) are communication with a linguistic alphabet. Two extra obligations appear.

**Listener metric.** Word error rate is a lab number. “Did the visitor understand her?” is Definition 9.1. Synthetic voice that sounds fluent can conceal wrong names.

**Latency as courtesy.** A three-second lag turns dialogue into dictation. Window hops and language-model context sit in the Chapter 4 budget whether or not the model is “real time.”

**Language and script.** A decoder-plus-prior trained on one language is not a universal larynx. For users whose language is under-represented in text models, S6 can *hurt* by pulling output toward the wrong tongue. Name the language on the design sheet.


---

## 9.6 Shared control and credit

> **Definition 9.3 (Credit split).**  
> A reported success rate accompanied by the fraction of trials in which S6 (prior, snap, safety layer, helper) changed the output.

Without Definition 9.3, a wheelchair that cannot hit a wall will look like a brilliant decoder. Without it, a speech system will look like cortex that attended grammar class.


In [ ]:
# Credit split cartoon
n = 100
decoder_correct = 62
s6_repaired = 21   # language model or snap fixed a wrong decode
helper_intervened = 8
failed = n - decoder_correct - s6_repaired - helper_intervened
print("headline '91 percent successful selections' would count", decoder_correct + s6_repaired + helper_intervened)
print("neural S5 alone                         ", decoder_correct)
print("S6 repair                               ", s6_repaired)
print("human helper                            ", helper_intervened)
print("failed                                  ", failed)
print("Definition 9.3 requires all four numbers.")


---

## 9.7 A day, costed

Write the day before you write the architecture.

| Block | Minutes | Notes |
|---|---|---|
| Setup and contact check | 15–40 | Caregiver time is her time |
| Calibration / refresh of $\theta$ | 0–35 | Definition 4.6 |
| Communication or control | variable | Only this block is usually published |
| Rest and artefact recovery | variable | Fatigue is not a footnote |
| Teardown | 5–15 | |

A system with ITR 20 bits/min and a 40-minute setup can lose to a board that is always on the tray. Publish the table or do not claim home use.


---

## 9.8 Worked choices for Amina

**Now, no surgery.** Test gaze and a muscle switch. If they fail, a P300 or SSVEP speller with a tiny yes/no fallback, language-model prior disclosed, undo cell, caregiver-mountable cap. Metric: corrected messages in two hours of household time.

**Later, trial.** Attempted speech if she consents to the resolution–risk point and to an owner in year five (Definition 3.4). Metric: understood utterances in conversation, lag stated.

**Not first.** A twelve-DOF arm demonstration as the communication plan. That is a different purpose theme.


---

## 9.9 What to carry into Chapter 10

Rehabilitation will reuse these loops with a different purpose: **improve** the nervous system, not only replace an output.

1. Metrics include the user’s clock.  
2. ITR is a comparison tool, not a life.  
3. S6 must be credited.  
4. Comparison devices get a fair trial.  
5. Language is a named design choice.


---

## 9.10 Chapter summary

Communication and motor substitution are closed loops judged by corrected meaning and completed tasks in a user’s day. Information transfer rate can compare two spellers and cannot replace setup time, undo policy, error cost, or a language-model prior. Speech substitution adds listener comprehension and conversational latency. Shared control and helpers must appear in the credit split. A fair design tests the non-BCI alternative first.


---

## Exercises

**9.1** Compute ITR for $N=36$, $P=0.8$, $T=10$ s and for $N=2$, $P=0.95$, $T=1.5$ s. Which would you give Amina for an emergency yes/no, and why is ITR the wrong decider?

**9.2** Write Definition 9.1 in forty of your own words and apply it to a paper that reports “12 characters per minute.”

**9.3** Design an undo that does not use an extra neural class. Place it on S0–S9.

**9.4** A speech system quotes 4 percent word error. List three questions that follow from Section 9.5.

**9.5** Produce a Definition 9.3 table for a wheelchair study in which software forbids speed above 0.3 m/s and a helper takes the joystick on 10 of 40 docks.

**9.6** Amina’s language is not the language of the pretrained prior. Describe one harm and one mitigation at S6.

**9.7** Fill the day-table for a wet-EEG P300 session at home. Invent plausible minutes; then compute how many published “characters per minute” survive as characters per *hour of household time*.

**9.8** Argue in 200 words that a successful laboratory reach-and-grasp is not tea-making. Use degrees of freedom and S6.


---

## Annotated reading

- Classic ITR notes in the BCI literature — learn the formula, then keep Section 9.2 beside it.  
- A home-use communication study (P300 or implant). Copy out whether setup time was reported.  
- A speech-BCI paper that includes a listener or conversation metric.  
- Assistive-technology comparison (gaze, switch, letter board) written by clinicians, not only by decoder authors.

---

*End of Chapter 9.*  
*Next: Chapter 10 — Rehabilitation and closed loops.*
